# Table of Contents
- 1. One-Hot and Dummy Encoding
- 2. Ordinal and Frequency Encoding
- 3. Target / Mean Encoding with Smoothing
- 4. Handling High-Cardinality Categoricals
- 5. Label Encoding (`LabelEncoder` for Target $y$ vs Features $X$)
- 6. 🎯 Critical Interview Traps, Master Decision Matrix & Viva Hacks

In [1]:
# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, LabelEncoder

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4)

print("All encoding libraries loaded successfully.")

All encoding libraries loaded successfully.


# 1. One-Hot and Dummy Encoding


In [2]:

'''
# vodafone,Airtel,Jio,BSNL
# how to do One hot encoding on this
             vodafone | Airtel | Jio | BSNL : are headings
vodafone    1         | 0      | 0    | 0
Airtel      0         | 1      | 0    | 0
Jio         0         | 0      | 1    | 0
BSNL        0         | 0      | 0    | 1


# let us eliminate the dummy variable Trap(usually the first column) to avoid multicollinearity
            
vodafone              0      | 0    | 0
Airtel                1      | 0    | 0
Jio                   0      | 1    | 0
BSNL                  0      | 0    | 1
'''
df_cat = pd.DataFrame({'color': ['Red', 'Blue', 'Green', 'Blue', 'Red']})
print("original dataframe:")
print(df_cat)
print("_" * 60)

df_cat_encoded = pd.get_dummies(df_cat, drop_first=True,dtype=int)
print("dataframe after one hot encoding (with dummy variable trap avoided):")
print(df_cat_encoded)


original dataframe:
   color
0    Red
1   Blue
2  Green
3   Blue
4    Red
____________________________________________________________
dataframe after one hot encoding (with dummy variable trap avoided):
   color_Green  color_Red
0            0          1
1            0          0
2            1          0
3            0          0
4            0          1


In [3]:
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Accessories'],
    'Price': [1200, 25, 75, 300, 80]
}

df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print("-" * 50)
encoded_cat = pd.get_dummies(df['Category'], prefix='Category', dtype=int, drop_first=True)
print("Encoded Category DataFrame:")
print(encoded_cat)
print("-" * 50)
'''
                Accessories  | Electronics
acessories  :       1        |    0
electronics :       0        |    1

_________________________________
Remove dummy variable for the first category to avoid multicollinearity.
                  | Electronics
acessories  :     |    0
electronics :     |    1
'''

# The resulting encoded_cat DataFrame will have only the 'Electronics' column, representing the presence of the 'Electronics' category.
# we will remove the Category column from the original DataFrame and concatenate the encoded_cat DataFrame
df = df.drop('Category', axis=1)
df = pd.concat([df, encoded_cat], axis=1)
print("DataFrame after encoding:")
print(df)
print("-" * 50)


Original DataFrame:
      Product     Category  Price
0      Laptop  Electronics   1200
1       Mouse  Accessories     25
2    Keyboard  Accessories     75
3     Monitor  Electronics    300
4  Headphones  Accessories     80
--------------------------------------------------
Encoded Category DataFrame:
   Category_Electronics
0                     1
1                     0
2                     0
3                     1
4                     0
--------------------------------------------------
DataFrame after encoding:
      Product  Price  Category_Electronics
0      Laptop   1200                     1
1       Mouse     25                     0
2    Keyboard     75                     0
3     Monitor    300                     1
4  Headphones     80                     0
--------------------------------------------------


In [4]:
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Accessories'],
    'Price': [1200, 25, 75, 300, 80]
}

df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print("-" * 50)


# lets cascade calls
df_dummies = pd.get_dummies(df['Category'], prefix='c_', dtype=int, drop_first=True)
df = pd.concat([df.drop('Category', axis=1), df_dummies], axis=1)
print("DataFrame after encoding:")
print(df)
print("-" * 50)


Original DataFrame:
      Product     Category  Price
0      Laptop  Electronics   1200
1       Mouse  Accessories     25
2    Keyboard  Accessories     75
3     Monitor  Electronics    300
4  Headphones  Accessories     80
--------------------------------------------------
DataFrame after encoding:
      Product  Price  c__Electronics
0      Laptop   1200               1
1       Mouse     25               0
2    Keyboard     75               0
3     Monitor    300               1
4  Headphones     80               0
--------------------------------------------------


In [5]:
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Accessories'],
    'Price': [1200, 25, 75, 300, 80]
}

df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print("-" * 50)


# # lets cascade calls
# df_dummies = pd.get_dummies(df['Category'], prefix='c_', dtype=int, drop_first=True)
# df = pd.concat([df.drop('Category', axis=1), df_dummies], axis=1)
# let us change commented code to scikitlearn
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', sparse_output=False,dtype=int)
encoded_cat = encoder.fit_transform(df[['Category']])
encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(['Category']))
df = pd.concat([df.drop('Category', axis=1), encoded_cat_df], axis=1)

print("DataFrame after encoding:")
print(df)
print("-" * 50)


Original DataFrame:
      Product     Category  Price
0      Laptop  Electronics   1200
1       Mouse  Accessories     25
2    Keyboard  Accessories     75
3     Monitor  Electronics    300
4  Headphones  Accessories     80
--------------------------------------------------
DataFrame after encoding:
      Product  Price  Category_Electronics
0      Laptop   1200                     1
1       Mouse     25                     0
2    Keyboard     75                     0
3     Monitor    300                     1
4  Headphones     80                     0
--------------------------------------------------


In [6]:
# 2. Ordinal and Frequency Encoding
# Ordinal encoding maps ordered categorical levels to sequential integers (reflecting inherent rank), 
# while frequency encoding replaces categories with their overall occurrence counts or relative frequencies.

df_ord = pd.DataFrame({'size': ['Small', 'Medium', 'Large', 'Medium', 'Small']})
mapping = {'Small': 1, 'Medium': 2, 'Large': 3}
df_ord['size_encoded'] = df_ord['size'].map(mapping)
print("DataFrame after ordinal encoding:")
print(df_ord)
print("-" * 50)

DataFrame after ordinal encoding:
     size  size_encoded
0   Small             1
1  Medium             2
2   Large             3
3  Medium             2
4   Small             1
--------------------------------------------------


In [7]:
df_edu = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Evan'],
    'Education': ['Bachelor', 'High School', 'PhD', 'Master', 'Bachelor']
})
print("Original DataFrame:")
print(df_edu)
print("-" * 50)

edu_mapping = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
df_edu['Education_encoded'] = df_edu['Education'].map(edu_mapping)
print("DataFrame after ordinal encoding:")
print(df_edu)
print("-" * 50)
# drop Education column
df_edu = df_edu.drop('Education', axis=1)
print("DataFrame after dropping Education column:")
print(df_edu)
print("-" * 50)

Original DataFrame:
      Name    Education
0    Alice     Bachelor
1      Bob  High School
2  Charlie          PhD
3    Diana       Master
4     Evan     Bachelor
--------------------------------------------------
DataFrame after ordinal encoding:
      Name    Education  Education_encoded
0    Alice     Bachelor                  2
1      Bob  High School                  1
2  Charlie          PhD                  4
3    Diana       Master                  3
4     Evan     Bachelor                  2
--------------------------------------------------
DataFrame after dropping Education column:
      Name  Education_encoded
0    Alice                  2
1      Bob                  1
2  Charlie                  4
3    Diana                  3
4     Evan                  2
--------------------------------------------------


In [8]:
data = {
    'Age': [25, 32, 47, 51, 23, 38],
    'Income': [50000, 75000, 120000, 110000, 40000, 85000],
    'Satisfaction': ['Medium', 'High', 'High', 'Low', 'Medium', 'Low'],
    'City': ['New York', 'London', 'Tokyo', 'New York', 'London', 'Tokyo']
}
df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print("-" * 60)
# reduce the data values
df['Age'] = (df['Age'] - df['Age'].mean())/df['Age'].std() # z-score
df['Income'] = (df['Income'] - df['Income'].mean())/df['Income'].std() # z-score
satisfaction_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df['Satisfaction'] = df['Satisfaction'].map(satisfaction_mapping)
df_city_encoded = pd.get_dummies(df['City'], prefix='City',dtype=int)
df = pd.concat([df, df_city_encoded], axis=1)
print(df)
print("-" * 60)

Original DataFrame:
   Age  Income Satisfaction      City
0   25   50000       Medium  New York
1   32   75000         High    London
2   47  120000         High     Tokyo
3   51  110000          Low  New York
4   23   40000       Medium    London
5   38   85000          Low     Tokyo
------------------------------------------------------------
        Age    Income  Satisfaction      City  City_London  City_New York  \
0 -0.960342 -0.943975             2  New York            0              1   
1 -0.349215 -0.157329             3    London            1              0   
2  0.960342  1.258634             3     Tokyo            0              0   
3  1.309557  0.943975             1  New York            0              1   
4 -1.134949 -1.258634             2    London            1              0   
5  0.174608  0.157329             1     Tokyo            0              0   

   City_Tokyo  
0           0  
1           0  
2           1  
3           0  
4           0  
5           1  
-

In [9]:
data = {
    'Employee': ['Alice', 'Bob', 'Charlie', 'Diana', 'Evan', 'Fiona', 'George', 'Hannah'],
    'Department': ['IT', 'HR', 'IT', 'Finance', 'HR', 'IT', 'Marketing', 'IT']
}
df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print("-" * 50)
department_counts = df['Department'].value_counts()
print("Counts per Department:")
print(department_counts)
print("-" * 50)
df['Department_Freq_Count'] = df['Department'].map(department_counts)
print(df)
print("-" * 50)


Original DataFrame:
  Employee Department
0    Alice         IT
1      Bob         HR
2  Charlie         IT
3    Diana    Finance
4     Evan         HR
5    Fiona         IT
6   George  Marketing
7   Hannah         IT
--------------------------------------------------
Counts per Department:
Department
IT           4
HR           2
Finance      1
Marketing    1
Name: count, dtype: int64
--------------------------------------------------
  Employee Department  Department_Freq_Count
0    Alice         IT                      4
1      Bob         HR                      2
2  Charlie         IT                      4
3    Diana    Finance                      1
4     Evan         HR                      2
5    Fiona         IT                      4
6   George  Marketing                      1
7   Hannah         IT                      4
--------------------------------------------------


In [10]:
# 3. Target / Mean Encoding with Smoothing
df_target = pd.DataFrame({
    'city': ['NY', 'SF', 'NY', 'CHI', 'SF'],
    'target': [1, 0, 1, 0, 1]
})
print("Target DataFrame:")
print(df_target)
print("-" * 50)

# Calculate the mean target for each city
city_target_mean = df_target.groupby('city')['target'].mean()
print("City Target Mean:")
print(city_target_mean)
print("-" * 50)

# Map the mean target to the original DataFrame
df_target['city_target_mean'] = df_target['city'].map(city_target_mean)
print("DataFrame with Target Encoding:")
print(df_target)
print("-" * 50)

Target DataFrame:
  city  target
0   NY       1
1   SF       0
2   NY       1
3  CHI       0
4   SF       1
--------------------------------------------------
City Target Mean:
city
CHI    0.0
NY     1.0
SF     0.5
Name: target, dtype: float64
--------------------------------------------------
DataFrame with Target Encoding:
  city  target  city_target_mean
0   NY       1               1.0
1   SF       0               0.5
2   NY       1               1.0
3  CHI       0               0.0
4   SF       1               0.5
--------------------------------------------------


In [11]:
'''
What is Handling High-Cardinality Categoricals?

Handling high-cardinality categoricals involves techniques to manage categorical features with a large number of unique values. 
High-cardinality features can lead to overfitting, increased memory usage, and computational inefficiency. Common strategies include:
- Frequency or count encoding
- Target or mean encoding
- Hashing
- Dimensionality reduction techniques

'''
df_high = pd.DataFrame({'sku': ['A', 'A', 'B', 'C', 'Rare1', 'Rare2', 'Rare3']})
print("Original High-Cardinality DataFrame:")
print(df_high)
print("-" * 50)
counts = df_high['sku'].value_counts()
print("SKU Counts:")
print(counts)
print("-" * 50)
frequent_skus = counts[counts > 1].index
print("Frequent SKUs:")
print(frequent_skus)
print("-" * 50)
df_high['sku_cleaned'] = df_high['sku'].apply(lambda x: x if x in frequent_skus else 'Other')
print("DataFrame with Cleaned SKUs:")
print(df_high)
print("-" * 50)


Original High-Cardinality DataFrame:
     sku
0      A
1      A
2      B
3      C
4  Rare1
5  Rare2
6  Rare3
--------------------------------------------------
SKU Counts:
sku
A        2
B        1
C        1
Rare1    1
Rare2    1
Rare3    1
Name: count, dtype: int64
--------------------------------------------------
Frequent SKUs:
Index(['A'], dtype='object', name='sku')
--------------------------------------------------
DataFrame with Cleaned SKUs:
     sku sku_cleaned
0      A           A
1      A           A
2      B       Other
3      C       Other
4  Rare1       Other
5  Rare2       Other
6  Rare3       Other
--------------------------------------------------


In [12]:
'''
what is cardinality?

Cardinality refers to the number of unique values in a categorical feature. 
High-cardinality features have many unique values, 
which can pose challenges for machine learning models, 
such as overfitting and increased computational complexity. 
Handling high-cardinality features often involves techniques like frequency encoding, 
target encoding, or grouping rare categories into an "Other" category.

Problems solved by handling high-cardinality features include:
- Reducing overfitting 
- Decreasing memory usage
- Improving computational efficiency
- Curse of dimensionality

'''
data = {
    'Order_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'City': ['New York', 'London', 'New York', 'Paris', 'London', 
             'Podunk', 'New York', 'BFE', 'Tokyo', 'Middleofnowhere']
}
df_city = pd.DataFrame(data)
print("Original City DataFrame:")
print(df_city)
print("-" * 50)

# Handling high-cardinality cities
counts_city = df_city['City'].value_counts()
frequent_cities = counts_city[counts_city > 1].index
df_city['City_cleaned'] = df_city['City'].apply(lambda x: x if x in frequent_cities else 'Other')
print("DataFrame with Cleaned Cities:")
print(df_city)
print("-" * 50)

# Displaying the counts of each city after cleaning
print("Counts of Cities After Cleaning:")
print(df_city['City_cleaned'].value_counts())
print("-" * 50)

Original City DataFrame:


   Order_ID             City
0       101         New York
1       102           London
2       103         New York
3       104            Paris
4       105           London
5       106           Podunk
6       107         New York
7       108              BFE
8       109            Tokyo
9       110  Middleofnowhere
--------------------------------------------------
DataFrame with Cleaned Cities:
   Order_ID             City City_cleaned
0       101         New York     New York
1       102           London       London
2       103         New York     New York
3       104            Paris        Other
4       105           London       London
5       106           Podunk        Other
6       107         New York     New York
7       108              BFE        Other
8       109            Tokyo        Other
9       110  Middleofnowhere        Other
--------------------------------------------------
Counts of Cities After Cleaning:
City_cleaned
Other       5
New York    3
London     

## 5. Label Encoding (`LabelEncoder` for Target $y$ vs Features $X$)

### What is Label Encoding?
`LabelEncoder` maps each unique category to an integer from $0$ to $K-1$:
$$\text{Cat} \mapsto 0, \quad \text{Dog} \mapsto 1, \quad \text{Elephant} \mapsto 2$$

---

### ⚠️ THE #1 INTERVIEW TRAP: `LabelEncoder` vs `OrdinalEncoder`
> **CRITICAL RULE:** `LabelEncoder` should **ONLY** be used for encoding the **Target variable ($y$)** in classification tasks!  
> It should **NEVER** be used to encode feature columns in **$X$**!

#### Why never use `LabelEncoder` on features $X$?
1. **Arbitrary Alphabetical Order:** `LabelEncoder` sorts categories alphabetically. It assigns numbers with no regard to true hierarchy.
2. **Hallucinated Math Relationships:** Linear models, SVMs, and neural networks treat integers as continuous numbers ($0 < 1 < 2$). If applied to nominal features (like `Red=0, Green=1, Blue=2`), the model assumes $\text{Blue} = 2 \times \text{Green}$, which is nonsense!
3. **Incompatible with Scikit-Learn Pipelines:** `LabelEncoder` only accepts 1D arrays `(n_samples,)`. Passing a 2D DataFrame `X[['city']]` to a `ColumnTransformer` will fail.
4. **Use `OrdinalEncoder` for Features:** `OrdinalEncoder` is specifically built for 2D feature matrices $X$ and accepts explicit ordered category hierarchies.

In [13]:
# 5. Label Encoding Example: Encoding Target Variable (y)
# Example: Predicting Customer Churn (Target: 'No Churn', 'Churn')
df_churn = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005],
    'tenure_months': [12, 3, 24, 6, 36],
    'churn_status': ['No Churn', 'Churn', 'No Churn', 'Churn', 'No Churn']
})

print("Original Dataset with Target Variable 'churn_status':")
print(df_churn)
print("-" * 50)

# Correct Use of LabelEncoder: On Target y (1D array)
le = LabelEncoder()
df_churn['churn_encoded'] = le.fit_transform(df_churn['churn_status'])

print("Target Encoded with LabelEncoder:")
print(df_churn[['customer_id', 'churn_status', 'churn_encoded']])
print("-" * 50)
print("LabelEncoder Classes:", le.classes_)
print("Inverse Transform [1, 0]:", le.inverse_transform([1, 0]))

Original Dataset with Target Variable 'churn_status':
   customer_id  tenure_months churn_status
0         1001             12     No Churn
1         1002              3        Churn
2         1003             24     No Churn
3         1004              6        Churn
4         1005             36     No Churn
--------------------------------------------------
Target Encoded with LabelEncoder:
   customer_id churn_status  churn_encoded
0         1001     No Churn              1
1         1002        Churn              0
2         1003     No Churn              1
3         1004        Churn              0
4         1005     No Churn              1
--------------------------------------------------
LabelEncoder Classes: ['Churn' 'No Churn']
Inverse Transform [1, 0]: ['No Churn' 'Churn']


## 6. 🎯 Critical Interview Traps, Master Decision Matrix & Viva Hacks

---

### 👑 The Master Categorical Encoding Decision Matrix

| Method | Target Variable ($y$) or Feature ($X$)? | Input Type | Cardinality | Output Cols | Preserves Natural Order? | Leakage Risk | Primary Scikit-Learn Tool |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **One-Hot Encoding** | Feature ($X$) | Nominal | Low ($< 10$) | $K$ | No | Zero | `OneHotEncoder(drop=None)` |
| **Dummy Encoding** | Feature ($X$) | Nominal | Low ($< 10$) | $K-1$ | No | Zero | `OneHotEncoder(drop='first')` |
| **Ordinal Encoding** | Feature ($X$) | Ordinal | Any | **1 Column** | **Yes ($1 < 2 < 3$)** | Zero | `OrdinalEncoder(categories=[...])` |
| **Frequency Encoding** | Feature ($X$) | Nominal | High ($> 20$) | **1 Column** | Implicit (Popularity) | Zero | `df['col'].value_counts(normalize=True)` |
| **Target Encoding** | Feature ($X$) | Nominal | High ($> 20$) | **1 Column** | Target Mean | **High (Needs OOF)** | `TargetEncoder(smooth='auto')` |
| **Label Encoding** | **Target ($y$) ONLY** | Target Class | Binary / Multi | **1 Column** | No (Arbitrary) | Zero | `LabelEncoder()` |

---

### 🛑 Top 5 Interview Traps

1. **`LabelEncoder` vs `OrdinalEncoder` Mistake:**
   - *Question:* "Can I use `LabelEncoder` on input feature columns?"
   - *Answer:* No! `LabelEncoder` expects a 1D target array $y$. Using it on $X$ introduces arbitrary integer ordering that biases linear models. Use `OrdinalEncoder` for features.

2. **The Dummy Variable Trap:**
   - In linear regression with an intercept $\beta_0$, using all $K$ one-hot columns causes perfect multicollinearity: $\sum x_i = 1$. The matrix $(X^T X)$ becomes non-invertible.
   - Fix: Always drop one column ($K-1$) via `drop='first'`.

3. **Catastrophic Target Leakage:**
   - Calculating Target Encoding before `train_test_split` leaks future test information into training rows.
   - Fix: Fit encoder exclusively on `X_train, y_train` and apply out-of-fold cross-validation.

4. **The Alphabetical Ordering Trap:**
   - By default, `OrdinalEncoder()` sorts strings alphabetically (`Bachelor -> 0, High School -> 1`), asserting Bachelor's is lower than High School!
   - Fix: Always pass explicit ordered lists: `categories=[['High School', 'Bachelor', 'Master', 'PhD']]`.

5. **Frequency Collisions:**
   - Two completely distinct categories with equal counts get mapped to the identical number. Combine with Top-N OHE or Target Encoding.